### Importaciones de librerias.
- La libreria SQLite3 para la conexion a nuestra base de datos.
- La libreria pandas para el manejo de los datos.
- La libreria numpy para el manejo de elementos numericos.
- La libreria matplotlib y seaborn para la graficacion de los datos.
- La libreria scikit-learn para el uso y entrenamiento de modelos de machine learning
- Tambien se instalo el plugin wrangler que facilitara la visualizacion de los datos para un mejor analisis de los mismos.

In [2]:
import sqlite3

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

### Conexion a la base de datos.

In [3]:
# Conexion a la base de datos.
conn = sqlite3.connect('database.sqlite')
query = "SELECT * FROM sqlite_master WHERE type = 'table';"

### Generar Data Frames

>Generar data frames a partir de las tablas existentes en la base de datos para realizar un analisis preliminar de los datos.

In [4]:
# Obtener todos los nombres de las tabas en la base de datos
tables = pd.read_sql_query(query, conn)['name'].tolist()

# Diccionario para almacernar todas las tablas en un dataframe.
df = {}

# Cargar cada tabla en un dataframe.
for table in tables:
    query = f"SELECT * FROM {table};"
    df[table] = pd.read_sql_query(query, conn)

# Cerrar la conexion a la base de datos.
conn.close()

# Mostrar los nombres de las tablas.
df.keys()


dict_keys(['sqlite_sequence', 'Player_Attributes', 'Player', 'Match', 'League', 'Country', 'Team', 'Team_Attributes'])

### Analisis Exploratorio de Datos (EDA)
> Exploracion de los datos: explorar la estructura del dataset para identificar tipos de datos y valores nulos.

In [5]:
df['sqlite_sequence'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   name    7 non-null      object
 1   seq     7 non-null      int64 
dtypes: int64(1), object(1)
memory usage: 244.0+ bytes


> La tabla "sqlite_sequence" cuenta solamente con 2 columnas y contiene un total de 7 registros completos sin nulos o valores atipicos en sus datos.
- cuenta con una (1) columna tipo string.
- cuenta con una (1) columna tipo entero.

In [6]:
df['Player_Attributes'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 183978 entries, 0 to 183977
Data columns (total 42 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   id                   183978 non-null  int64  
 1   player_fifa_api_id   183978 non-null  int64  
 2   player_api_id        183978 non-null  int64  
 3   date                 183978 non-null  object 
 4   overall_rating       183142 non-null  float64
 5   potential            183142 non-null  float64
 6   preferred_foot       183142 non-null  object 
 7   attacking_work_rate  180748 non-null  object 
 8   defensive_work_rate  183142 non-null  object 
 9   crossing             183142 non-null  float64
 10  finishing            183142 non-null  float64
 11  heading_accuracy     183142 non-null  float64
 12  short_passing        183142 non-null  float64
 13  volleys              181265 non-null  float64
 14  dribbling            183142 non-null  float64
 15  curve            

In [26]:
df['Player_Attributes'].describe()

,id,player_fifa_api_id,player_api_id,overall_rating,potential,crossing,finishing,heading_accuracy,short_passing,volleys,...,vision,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes
count,183978.00000,183978.000000,183978.000000,183142.000000,183142.000000,183142.000000,183142.000000,183142.000000,183142.000000,181265.000000,...,181265.000000,183142.000000,183142.000000,183142.000000,181265.000000,183142.000000,183142.000000,183142.000000,183142.000000,183142.000000
mean,91989.50000,165671.524291,135900.617324,68.600015,73.460353,55.086883,49.921078,57.266023,62.429672,49.468436,...,57.873550,55.003986,46.772242,50.351257,48.001462,14.704393,16.063612,20.998362,16.132154,16.441439
std,53110.01825,53851.094769,136927.840510,7.041139,6.592271,17.242135,19.038705,16.488905,14.194068,18.256618,...,15.144086,15.546519,21.227667,21.483706,21.598778,16.865467,15.867382,21.452980,16.099175,17.198155
min,1.00000,2.000000,2625.000000,33.000000,39.000000,1.000000,1.000000,1.000000,3.000000,1.000000,...,1.000000,2.000000,1.000000,1.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,45995.25000,155798.000000,34763.000000,64.000000,69.000000,45.000000,34.000000,49.000000,57.000000,35.000000,...,49.000000,45.000000,25.000000,29.000000,25.000000,7.000000,8.000000,8.000000,8.000000,8.000000
50%,91989.50000,183488.000000,77741.000000,69.000000,74.000000,59.000000,53.000000,60.000000,65.000000,52.000000,...,60.000000,57.000000,50.000000,56.000000,53.000000,10.000000,11.000000,12.000000,11.000000,11.000000
75%,137983.75000,199848.000000,191080.000000,73.000000,78.000000,68.000000,65.000000,68.000000,72.000000,64.000000,...,69.000000,67.000000,66.000000,69.000000,67.000000,13.000000,15.000000,15.000000,15.000000,15.000000
max,183978.00000,234141.000000,750584.000000,94.000000,97.000000,95.000000,97.000000,98.000000,97.000000,93.000000,...,97.000000,96.000000,96.000000,95.000000,95.000000,94.000000,93.000000,97.000000,96.000000,96.000000


> La tabla "Player_Attributes" cuenta con 42 columnas y 183978 registros en su totalidad, sin embargo se pueden observar diversos valores nulos en diferentes columnas, la columna que contiene fechas esta como texto, se evaluara como proceder con su tratamiento de los datos faltantes y la columna fechas.

- cuenta con treinta y cinco (35) columnas tipo Flotantes.
- cuenta con tres (3) columnas tipo entero.
- cuenta con cuatro (4) columnas tipo texto.

In [16]:
df['Player'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11060 entries, 0 to 11059
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  11060 non-null  int64  
 1   player_api_id       11060 non-null  int64  
 2   player_name         11060 non-null  object 
 3   player_fifa_api_id  11060 non-null  int64  
 4   birthday            11060 non-null  object 
 5   height              11060 non-null  float64
 6   weight              11060 non-null  int64  
dtypes: float64(1), int64(4), object(2)
memory usage: 605.0+ KB


> La tabla "Player" cuenta con 7 columnas y 11060 registros registros completos sin nulos, la columna fechas se encuentra en un formato de texto.

- cuenta con una (1) columna tipo Flotante.
- cuenta con  cuatro (4) columnas tipo entero.
- cuenta con dos (2) columnas tipo texto.

In [24]:
df['Match'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25979 entries, 0 to 25978
Columns: 115 entries, id to BSA
dtypes: float64(96), int64(9), object(10)
memory usage: 22.8+ MB


> La tabla "Match" cuenta con 115 columnas y 25979 registros en su totalidad, sin embargo se pueden observar diversos valores nulos en diferentes columnas, existen dos columnas que contienen fechas y se encuentran en formato de tipo texto, tambien se observario registros atipicos 8 de las columnas, se evaluara como proceder con su tratamiento de los datos faltantes, valores atipicos y columnas fechas.


- cuenta con noventa y seis (96) columna tipo Flotante.
- cuenta con  nueve (9) columnas tipo entero.
- cuenta con diez (10) columnas tipo texto.

In [9]:
df['League'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          11 non-null     int64 
 1   country_id  11 non-null     int64 
 2   name        11 non-null     object
dtypes: int64(2), object(1)
memory usage: 396.0+ bytes


> La tabla "League" cuenta con 3 columnas y 11 registros en su totalidad, sin nulos.



- cuenta con  dos (2) columnas tipo entero.
- cuenta con una (1) columna tipo texto.

In [10]:
df['Country'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      11 non-null     int64 
 1   name    11 non-null     object
dtypes: int64(1), object(1)
memory usage: 308.0+ bytes


In [11]:
df['Team'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                299 non-null    int64  
 1   team_api_id       299 non-null    int64  
 2   team_fifa_api_id  288 non-null    float64
 3   team_long_name    299 non-null    object 
 4   team_short_name   299 non-null    object 
dtypes: float64(1), int64(2), object(2)
memory usage: 11.8+ KB


In [12]:
df['Team_Attributes'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458 entries, 0 to 1457
Data columns (total 25 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              1458 non-null   int64  
 1   team_fifa_api_id                1458 non-null   int64  
 2   team_api_id                     1458 non-null   int64  
 3   date                            1458 non-null   object 
 4   buildUpPlaySpeed                1458 non-null   int64  
 5   buildUpPlaySpeedClass           1458 non-null   object 
 6   buildUpPlayDribbling            489 non-null    float64
 7   buildUpPlayDribblingClass       1458 non-null   object 
 8   buildUpPlayPassing              1458 non-null   int64  
 9   buildUpPlayPassingClass         1458 non-null   object 
 10  buildUpPlayPositioningClass     1458 non-null   object 
 11  chanceCreationPassing           1458 non-null   int64  
 12  chanceCreationPassingClass      14

In [20]:
df['sqlite_sequence']

,name,seq
0,Team,103916
1,Country,51958
2,League,51958
3,Match,51958
4,Player,11075
5,Player_Attributes,183978
6,Team_Attributes,1458
